In [ ]:
import torch
from transformers import AutoTokenizer
from arabert.preprocess import ArabertPreprocessor
# for text classification
from transformers import AutoModelForSequenceClassification

## araBERT Preprocessor

In [ ]:
model_name="bert-base-arabert"
arabert_prep = ArabertPreprocessor(model_name=model_name)

|## Initializing Model

In [ ]:
checkpoint = "aubmindlab/bert-base-arabert"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, return_dict=True)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
def classify(text):
    text = arabert_prep.preprocess(text)
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=1)
    return probs

## Inference

In [ ]:
def classify(text):
    text = arabert_prep.preprocess(text)
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=1)
    return probs

In [ ]:
text = 'هذا مثال عربي اصطناعي لاختبار معالجة النصوص.'
probs = classify(text)
print(probs)

# Training Setup

## Dataset Prep

### AI

In [ ]:
import pandas as pd
# دمج الملفات فقط
# List of the file paths for the CSV files you want to merge
csv_files = ['chatgpt_Tweets01.csv', 'Gemeni_Tweets01.csv']

# Initialize an empty list to store the transformed DataFrames
transformed_dfs = []

# Loop through the list of files
for file in csv_files:
    # Read the current CSV file into a DataFrame
    df = pd.read_csv(file, encoding='utf-8')  # Adjust encoding if necessary

    # Concatenate all columns into one column named 'text'
    # The assumption here is that all values are string; if not, you might need to cast them
    # Use .astype(str) if there are non-string types
    concatenated_column = df.apply(lambda row: ' '.join(row.values.astype(str)), axis=1)

    # Create a new DataFrame with this single concatenated column
    transformed_df = pd.DataFrame(concatenated_column, columns=['text'])

    # Append the transformed DataFrame to the list
    transformed_dfs.append(transformed_df)

# Concatenate all transformed DataFrames in the list into a single DataFrame
merged_df = pd.concat(transformed_dfs, ignore_index=True)

# Save the merged DataFrame to a new CSV file
merged_df.to_csv('test_Tweets01.csv', index=False, encoding='utf-8')



In [ ]:
import pandas as pd
import re

def clean_arabic_text(text):
    if not isinstance(text, str):
        return ""

    # 1. إزالة الروابط (URLs)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 2. إزالة المنشن والهاشتاج (ابقاء الكلمة وحذف الرمز #)
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'#', '', text)
    
    # 3. توحيد الحروف (Normalization)
    # تحويل (إ، أ، آ) إلى (ا) | تحويل (ة) إلى (ه) | تحويل (ى) إلى (ي)
    text = re.sub(r'[إأآ]', 'ا', text)
    
    # 4. إزالة التطويل (المد) مثل: "جمييييييل" تصبح "جميل"
    text = re.sub(r'(.)\1+', r'\1\1', text) # يبقي على حرفين فقط كحد أقصى
    
    # 5. إزالة الرموز التعبيرية (Emojis) والرموز غير العربية
    # سنبقي فقط على الحروف العربية والمسافات
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    
    # 6. إزالة المسافات الزائدة
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# تحميل الملف المدمج
df = pd.read_csv('final_dataset.csv')

print("بدء عملية التنظيف... قد تستغرق دقيقة نظراً لحجم البيانات")

# تطبيق الدالة على عمود النص
df['text_cleaned'] = df['text'].apply(clean_arabic_text)

# حذف الصفوف التي قد تصبح فارغة بعد التنظيف (إن وجدت)
df = df[df['text_cleaned'] != ""]

# حفظ الملف المنظف الجديد
df[['text_cleaned', 'label']].to_csv('cleaned_dataset.csv', index=False, encoding='utf-8-sig')

print("تم التنظيف بنجاح! الملف جاهز الآن للتدريب باسم: cleaned_dataset.csv")
print(df[['text', 'text_cleaned']].head())

In [ ]:
import pandas as pd
import glob
import os

# 1. تحديد المسار الذي توجد به الملفات (نستخدم '*' لجلب كل ملفات csv)
path = './' # تعني المجلد الحالي
all_files = glob.glob(os.path.join(path, "*.csv"))

# التأكد من عدد الملفات المكتشفة
print(f"تم العثور على {len(all_files)} ملفات للدمج.")

# 2. قراءة ودمج الملفات
li = []

for filename in all_files:
    # نتخطى الملف النهائي إذا كان موجوداً مسبقاً لكي لا ندمجه مع نفسه
    if filename.endswith('final_dataset.csv'):
        continue
        
    try:
        # قراءة الملف مع تحديد الترميز
        df = pd.read_csv(filename, index_col=None, header=0)
        li.append(df)
        print(f"تمت إضافة: {filename}")
    except Exception as e:
        print(f"خطأ في قراءة {filename}: {e}")

# 3. دمج القائمة في DataFrame واحد
merged_df = pd.concat(li, axis=0, ignore_index=True)

# 4. خطوة اختيارية ومهمة جداً للتدريب: خلط البيانات (Shuffling)
# لضمان عدم وجود أنماط متكررة من نفس المصدر وراء بعضها
merged_df = merged_df.sample(frac=1).reset_index(drop=True)

# 5. حفظ النتيجة النهائية
merged_df.to_csv('final_dataset.csv', index=False, encoding='utf-8-sig')

print("-" * 30)
print(f"تم الدمج بنجاح! إجمالي الصفوف: {len(merged_df)}")
print(merged_df['label'].value_counts())

In [ ]:
import pandas as pd
#تسميه العادمود الاول الى "text" 
df = pd.read_csv('chatgptAR.csv')
# Assuming df is your DataFrame
# Renaming the first column to "text"
df.rename(columns={df.columns[0]: "text"}, inplace=True)

# If you then want to save this DataFrame to a CSV file with the updated column name
df.to_csv('chatgpt_Tweets.csv', index=False, encoding='utf-8')


In [ ]:
import pandas as pd
#يحذف العامود الثاني
# Load the CSV file
file_path = 'test_Tweets11111.csv'  # Replace with your file path
data = pd.read_csv(file_path)

# Remove the second column using its index
data_cleaned = data.drop(data.columns[1], axis=1)  # Indexes start at 0, so 1 is the second column

# Save the cleaned data to a new CSV file
cleaned_file_path = 'test_Tweets999999.csv'  # Replace with your desired save path
data_cleaned.to_csv(cleaned_file_path, index=False)

print("Second column removed and data saved to:", cleaned_file_path)



In [ ]:
import pandas as pd
#وضع عامود label 0
# Path to your CSV file
csv_file_path = 'chatgpt_Tweets.csv'

# Reading the CSV file into a DataFrame
df = pd.read_csv(csv_file_path, encoding='utf-8')  # Adjust encoding if necessary

# Adding a new column 'label' with all values set to 0
df['label'] = 0

# Save the updated DataFrame back to a CSV file, with the new column
df.to_csv('chatgpt_Tweets01.csv', index=False, encoding='utf-8')



In [ ]:
import pandas as pd
# حذف التكرار في عامود "text"
# Load the CSV file
df = pd.read_csv('chatgpt_Tweets01.csv')

# Drop duplicates in the 'text' column
df.drop_duplicates(subset='text', keep='first', inplace=True)

# Save the result to a new CSV file
df.to_csv('chatgpt_Tweets01u.csv', index=False)


In [ ]:
import pandas as pd
#حذف مشكلة رقم 1 في نهاية الخلية
# Load the CSV file
df = pd.read_csv('data_tweets01.csv')

# Function to remove '1' from the end of a string
def remove_ending_one(text):
    if pd.isna(text):
        return text
    return text[:-1] if text.endswith('1') else text

# Apply the function to the 'text' column
df['text'] = df['text'].apply(remove_ending_one)

# Save the modified DataFrame back to CSV
df.to_csv('data_tweets01u.csv', index=False)


In [ ]:
import pandas as pd
from sklearn.utils import resample, shuffle
#يختار عدد خلايا مساوية لجميع الليبل 
# Load the data
data = pd.read_csv('data_tweets05_6Words.csv')

# Count the number of rows with label '0' and select the same number of rows with label '1'
label_0_data = data[data['label'] == 0]
label_1_data = data[data['label'] == 1]
selected_label_1 = resample(label_1_data, n_samples=label_0_data.shape[0], random_state=42, replace=False)

# Combine the selected label '1' rows with all label '0' rows
combined_data = pd.concat([label_0_data, selected_label_1])

# Shuffle the combined dataset
shuffled_data = shuffle(combined_data, random_state=42)

# Save the shuffled data to a new file
shuffled_data_path = 'data_tweets05_balanced_6Words.csv'
shuffled_data.to_csv(shuffled_data_path, index=False)

print(f"Data saved to {shuffled_data_path}")


In [ ]:
import pandas as pd
from sklearn.utils import resample
# for dataset testing
# يختار خلايا بشكل عشوائي ثم يقوم بحذفها من الملف الاصلي 
# Load the data
data = pd.read_csv('data_tweets01_6Words.csv')

# Select a specific number of rows with label '1'
label_1_data = data[data['label'] == 1]
selected_label_1 = resample(label_1_data, n_samples=1852, random_state=42, replace=False)

# Save the selected label '1' rows to a new file
selected_label_1_path = 'data_tweets05_T6WORDS.csv'
selected_label_1.to_csv(selected_label_1_path, index=False)

# Remove the selected rows from the original data
data_remaining = data.drop(selected_label_1.index)

# Save the modified original data to a new file
modified_data_path = 'data_tweets05_6WordsTTT.csv'
data_remaining.to_csv(modified_data_path, index=False)

print(f"Selected rows saved to {selected_label_1_path}")
print(f"Modified original data saved to {modified_data_path}")



In [ ]:
import pandas as pd
#حذف النقاط وعلامة التنصيص
# Load the CSV file
file_path = 'data_tweets05555_testing.csv'  # Replace with your file path
data = pd.read_csv(file_path)

# Remove periods and quotation marks from the 'text' column
data['text'] = data['text'].str.replace('.', '', regex=False)
data['text'] = data['text'].str.replace('"', '', regex=False)

# Save the cleaned data to a new CSV file
cleaned_file_path = 'data_tweets05555_testing9.csv'  # Replace with your desired save path
data.to_csv(cleaned_file_path, index=False)

print("Periods and quotation marks removed. Data saved to:", cleaned_file_path)


In [ ]:
import pandas as pd
#حذف الحركات 
# Load the CSV file
file_path = 'data_tweets05555_testing9.csv'  # Replace with your file path
data = pd.read_csv(file_path)

# List of Arabic diacritics to remove
diacritics = ['َ', 'ُ', 'ِ', 'ْ']

# Remove the Arabic diacritics from the 'text' column
for diacritic in diacritics:
    data['text'] = data['text'].str.replace(diacritic, '', regex=False)

# Save the cleaned data to a new CSV file
cleaned_file_path = 'data_tweets05555_testing99.csv'  # Replace with your desired save path
data.to_csv(cleaned_file_path, index=False)

print("Arabic diacritics removed. Data saved to:", cleaned_file_path)


In [ ]:
import pandas as pd
import re
#حذف علامة ! و علامة ؟ اذا تكررت اكثر من مره 
# Load the CSV file
file_path = 'data_tweets05555_testing99.csv'  # Replace with your file path
data = pd.read_csv(file_path)

# Function to replace consecutive characters
def replace_consecutive_chars(text, char):
    pattern = re.escape(char) + r'{2,}'  # Matches the character repeated two or more times
    return re.sub(pattern, char, text)

# Replace consecutive "!" and "?"
data['text'] = data['text'].apply(lambda x: replace_consecutive_chars(x, '!'))
data['text'] = data['text'].apply(lambda x: replace_consecutive_chars(x, '?'))

# Save the cleaned data to a new CSV file
cleaned_file_path = 'data_tweets05555_testing999.csv'  # Replace with your desired save path
data.to_csv(cleaned_file_path, index=False)

print("Consecutive '!' and '?' characters reduced to one. Data saved to:", cleaned_file_path)


In [ ]:
import pandas as pd
#حذف علامة ! و علامة ؟ ه 
# Load the CSV file
file_path = 'data_tweets05555_testing999.csv'  # Replace with your file path
data = pd.read_csv(file_path)

# Remove "!" and "?" from the 'text' column
data['text'] = data['text'].str.replace('!', '', regex=False)
data['text'] = data['text'].str.replace('?', '', regex=False)

# Save the cleaned data to a new CSV file
cleaned_file_path = 'data_tweets05555_testing9999.csv'  # Replace with your desired save path
data.to_csv(cleaned_file_path, index=False)

print("Exclamation marks and question marks removed. Data saved to:", cleaned_file_path)


In [ ]:
import pandas as pd
import re
#حذف الاموجي
# Load the dataset
file_path = 'data_tweets05555_testing9999.csv'
data = pd.read_csv(file_path)

# Define the regex pattern to match emojis
emoji_pattern = re.compile("["
                           "\U0001F600-\U0001F64F"  # emoticons
                           "\U0001F300-\U0001F5FF"  # symbols & pictographs
                           "\U0001F680-\U0001F6FF"  # transport & map symbols
                           "\U0001F700-\U0001F77F"  # alchemical symbols
                           "\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
                           "\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
                           "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
                           "\U0001FA00-\U0001FA6F"  # Chess Symbols
                           "\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
                           "\U00002702-\U000027B0"  # Dingbats
                           "\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)

# Function to remove emojis and count them
def remove_and_count_emojis(text):
    count = len(emoji_pattern.findall(text))
    cleaned_text = emoji_pattern.sub('', text)
    return cleaned_text, count

# Apply the function and count total emojis
data['text'], emoji_counts = zip(*data['text'].apply(remove_and_count_emojis))
total_emojis_deleted = sum(emoji_counts)

# Display the number of emojis deleted
print(f"Total emojis deleted: {total_emojis_deleted}")

# Save the cleaned data
data.to_csv('data_tweets05_testing.csv', index=False)



In [ ]:
import pandas as pd
#يختار الخلايا اللتي اكبر من 6
# Load the CSV file
file_path = 'data_tweets01.csv'  # Replace with your file path
data = pd.read_csv(file_path)

# Define a function to count words in a string, handling cases where the text might not be a string
def word_count(text):
    if pd.isna(text):
        return 0  # Return 0 if text is NaN
    return len(text.split())

# Filter rows where the 'text' column has fewer than 6 words
data_filtered = data[data['text'].apply(word_count) >= 7]

# Save the cleaned data to a new CSV file
cleaned_file_path = 'data_tweets01_8Words.csv'  # Replace with your desired save path
data_filtered.to_csv(cleaned_file_path, index=False)

print("Rows with fewer than 6 words in 'text' column removed. Data saved to:", cleaned_file_path)



In [ ]:
import pandas as pd
#عرض عدد الخلايا 
# Load the CSV file
df = pd.read_csv('data_tweets01_8Words_balanced.csv')

# Total number of cells in the DataFrame
total_cells = df.size

# Count of non-empty cells in the DataFrame
non_empty_cells = df.count().sum()

# Count of unique values in a specific column, e.g., 'text'
unique_values_in_column = df['text'].nunique()

print("Total cells:", total_cells)
print("Non-empty cells:", non_empty_cells)
print("Unique values in 'text' column:", unique_values_in_column)


In [ ]:
import pandas as pd
#خذف علامة = ا
# Load the CSV file
file_path = 'data_tweets00.csv'  # Replace with your file path
data = pd.read_csv(file_path)

# Remove "=" signs from the 'text' column
data['text'] = data['text'].str.replace('=', '', regex=False)

# Save the cleaned data to a new CSV file
cleaned_file_path = 'data_tweets0099.csv'  # Replace with your desired save path
data.to_csv(cleaned_file_path, index=False)

print("Equal signs removed. Data saved to:", cleaned_file_path)


In [ ]:
import pandas as pd
#وضع عامود label 1 

# Load your CSV file
df = pd.read_csv('HR clean.csv')

# Add a column titled 'Label' with all values set to 1
df['Label'] = 1

# Save the modified DataFrame back to a CSV file
df.to_csv('modified_file1.csv', index=False)


In [ ]:
import pandas as pd
#دمج الملفات
# List of file paths you want to merge and shuffle
file_paths = [
    'chatgpt_Tweets01.csv','Gemeni_Tweets01.csv']

# Initialize an empty list to store DataFrames
dfs = []

# Iterate over each file path, read the file into a DataFrame, and append it to the list
for file_path in file_paths:
    df = pd.read_csv(file_path, encoding='utf-8')
    dfs.append(df)

# Concatenate all DataFrames in the list into a single DataFrame
merged_df = pd.concat(dfs, ignore_index=True)

# Shuffle the merged DataFrame rows randomly
merged_df_shuffled = merged_df.sample(frac=1).reset_index(drop=True)

# Specify the path where you want to save the merged and shuffled CSV
output_path = 'test_Tweets01.csv'

# Save the DataFrame to a CSV file
merged_df_shuffled.to_csv(output_path, index=False, encoding='utf-8')

print(f'Merged and shuffled DataFrame saved to {output_path}')


In [ ]:
import pandas as pd
import numpy as np
#دمج ملف  
# Assuming 'B.csv' is in the correct path and accessible
ai_df = pd.read_csv('test_Tweets01.csv')
# Check the number of columns in ai_df and adjust the columns assignment accordingly
print("Number of columns in ai_df:", len(ai_df.columns))
ai_df.columns = ['text', 'label']  # Adjust this line if the number of columns is not 2

# Set all the labels to 0
ai_df['label'] = 0
ai_df = ai_df.drop(0)

# Load the 'Booking Hotel Reviews.csv' with correct path
human_df = pd.read_csv('data_tweets05_T6WORDS.csv')
human_df.columns = ['text', 'label']
# Set all the labels to 1
human_df['label'] = 1

# Merge the two dataframes
df = pd.concat([ai_df, human_df], ignore_index=True)
df = df.sample(frac=1).reset_index(drop=True)
# Drop nan values
df = df.dropna()

# Save the dataframe to a csv file
df.to_csv('data_tweets05555_testing.csv', index=False)


## Data Preprocessing

In [ ]:



from datasets import load_dataset

dataset = load_dataset("csv", data_files="HR_clean_AiH_no_emojis.csv", split="train" )

dataset = dataset.train_test_split(test_size=0.3)



In [ ]:
print(dataset)

In [ ]:
dataset["train"]

In [ ]:
dataset["train"][0]

### Applying araBERT Preprocessor

In [ ]:
def preprocess_function(examples):
    arabic_prep = arabert_prep.preprocess(examples["text"])
    result = tokenizer(arabic_prep,truncation=True,   
                       max_length=512, return_overflowing_tokens=True)

    sample_map = result.pop("overflow_to_sample_mapping")
    for key, values in examples.items():
        result[key] = [values[i] for i in sample_map]
    return result
    # return tokenizer(arabic_prep, truncation=True, max_length=512, padding="max_length")

In [ ]:
tokenized_dataset = dataset.map(preprocess_function, batched=True)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Setup Evaluation

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
def calculate_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    acc = accuracy.compute(predictions=predictions, references=labels)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')

    return {"accuracy": acc["accuracy"], "f1": f1, "recall": recall, "precision": precision}

## Training

In [ ]:
# find number of labels
num_labels = len(set(dataset["train"]["label"]))
print(num_labels)

# get label names
label_names = ["AI", "Human"]
print(label_names)

In [ ]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in id2label.items()}

In [ ]:
print(id2label)
print(label2id)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=num_labels, id2label=id2label, label2id=label2id
)

In [ ]:
batch_size = 1
exp = "1"
training_args = TrainingArguments(
    output_dir="./trained/araBERT-base"+"_exp"+exp,
    learning_rate=2e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=10,
    save_total_limit=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    fp16=False,
    logging_dir="./logs/araBERT-base"+"_exp"+exp,
    logging_strategy = "epoch",
    logging_steps = 1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=calculate_metrics,
)

In [ ]:
 trainer.train()

## Final Inference

In [ ]:
from datasets import load_dataset
import torch

dataset = load_dataset("csv", data_files="HR_clean_AiH_no_emojis.csv", split="train")

dataset = dataset.train_test_split(test_size=0.2)

In [ ]:
from transformers import AutoTokenizer
model_path = "trained/araBERT-base_exp1/checkpoint-206649"
tokenizer = AutoTokenizer.from_pretrained(model_path)

text = dataset["train"][0]["text"]

inputs = tokenizer(text, return_tensors="pt")

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs.to(device)

with torch.no_grad():
    logits = model(**inputs).logits

In [ ]:
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]

### Evaluation

In [ ]:
from tqdm.notebook import tqdm
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from torch.utils.data import DataLoader

def evaluate_test_set(model, tokenizer, test_dataset, batch_size=32, pbar=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    # Metrics storage
    overall_accuracy = []
    metrics = {0: {'precision': [], 'recall': [], 'f1': []},
               1: {'precision': [], 'recall': [], 'f1': []}}

    test_dataloader = DataLoader(test_dataset, batch_size=batch_size)
    if pbar:
        progress = tqdm(total=len(test_dataloader), desc='Evaluating')

    for batch in test_dataloader:
        inputs = tokenizer(batch["text"], padding=True, truncation=True, return_tensors="pt", max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        labels = batch["label"].to(device)
        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        acc = accuracy_score(labels.cpu().numpy(), predictions.cpu().numpy())
        overall_accuracy.append(acc)

        for label in [0, 1]:
            label_precision = precision_score(labels.cpu(), predictions.cpu(), labels=[label], average='binary', zero_division=0)
            label_recall = recall_score(labels.cpu(), predictions.cpu(), labels=[label], average='binary', zero_division=0)
            label_f1 = f1_score(labels.cpu(), predictions.cpu(), labels=[label], average='binary', zero_division=0)

            metrics[label]['precision'].append(label_precision)
            metrics[label]['recall'].append(label_recall)
            metrics[label]['f1'].append(label_f1)

        if pbar:
            progress.update(1)

    if pbar:
        progress.close()

    # Calculate average metrics
    avg_accuracy = np.mean(overall_accuracy)
    for label in metrics:
        for metric in metrics[label]:
            metrics[label][metric] = np.mean(metrics[label][metric])

    return avg_accuracy, metrics


In [ ]:
for example in dataset["test"]:
    print(example["text"])
    print(example["label"])
    break

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "aubmindlab/bert-base-arabertv2"
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
accuracy, f1, recall, precision =  evaluate_test_set(model, tokenizer, dataset["test"], batch_size=256)

In [ ]:
avg_accuracy, label_metrics = evaluate_test_set(model, tokenizer, test_dataset, batch_size=32, pbar=True)

print(f"Overall Accuracy: {avg_accuracy*100:.2f}%")
for label in label_metrics:
    print(f"Metrics for label {label}:")
    print(f"- Precision: {label_metrics[label]['precision']*100:.2f}%")
    print(f"- Recall: {label_metrics[label]['recall']*100:.2f}%")
    print(f"- F1 Score: {label_metrics[label]['f1']*100:.2f}%\n")


In [ ]:
import pandas as pd
from collections import Counter

# ضع مسار الملف هنا
file_path = 'مسار_ملفك_هنا.csv'

# قم بتحميل البيانات، استبدل 'windows-1256' بالترميز الصحيح لملفك إذا لزم الأمر
data = pd.read_csv(r"data/input.csv", encoding='windows-1256')


# دمج كل النصوص الموجودة في العمود 'text' إلى نص واحد كبير
all_text = ' '.join(data['text'])

# فصل النص إلى كلمات
words = all_text.split()

# حساب تكرار كل كلمة
word_counts = Counter(words)

# الحصول على الكلمة الأكثر تكرارًا وعدد التكرارات
most_common_word, count = word_counts.most_common(1)[0]

print(f"أكثر كلمة تكرارًا هي: '{most_common_word}' وتكررت {count} مرة/مرات.")


In [ ]:
import pandas as pd
from collections import Counter

# تعريف مسار الملف
file_path = r"data/input.csv" # استبدل بمسار الملف الفعلي

# قراءة الملف مع تحديد الترميز إذا لزم الأمر
# إذا واجهت مشكلة في الترميز يمكنك تجربة 'utf-8' أو 'windows-1256'
try:
    data = pd.read_csv(file_path, encoding='windows-1256')
except Exception as e:
    print(f"حدث خطأ أثناء قراءة الملف: {e}")
    exit()

# دمج كل النصوص الموجودة في عمود "text" إلى نص واحد كبير
all_texts = ' '.join(data['text'])

# تقسيم النص الكبير إلى كلمات باستخدام الفراغات
words = all_texts.split()

# حساب تكرار كل كلمة واسترجاع أكثر عشرين كلمة تكرارًا
most_common_words = Counter(words).most_common(20)

# طباعة أكثر عشرين كلمة تكرارًا
for word, frequency in most_common_words:
    print(f"الكلمة: '{word}', تكررت: {frequency} مرة/مرات")
